# 面试问题：目标检测中的 IoU、NMS 和 mAP 如何从零实现？

可以直接复述的回答是：IoU 是预测框与真值框交集面积除以并集面积。NMS 按 score 降序保留框，并抑制同类别中与已保留框 IoU 超阈值的重复预测。类别无关 NMS 可能误删重叠的不同对象，例如人和背包。评估时每个预测只能匹配同图同类且尚未匹配的一个 GT，按全局 score 排序累积 precision/recall，再对类别 AP 求均值得到 mAP。重复框即使位置准确也是 false positive。下面用六幅图的真实框坐标手写完整流程。

## 真实案例：六幅仓储图中的人员与背包检测

框采用 `[x1,y1,x2,y2]` 半开坐标。第一幅图的人与背包高度重叠，用于复现类别无关 NMS 误删；多幅图还有高分重复框，用于观察 NMS 对 AP 的影响。

In [1]:
ground_truth = {  # 定义六幅图的人员与背包真值框
    "DET-01": [{"class": "person", "box": [0, 0, 6, 8]}, {"class": "bag", "box": [1, 1, 6, 7]}],  # 人与背包高度重叠
    "DET-02": [{"class": "person", "box": [1, 0, 5, 8]}],  # 单人场景
    "DET-03": [{"class": "bag", "box": [2, 2, 6, 6]}],  # 单背包场景
    "DET-04": [{"class": "person", "box": [0, 1, 4, 8]}, {"class": "bag", "box": [3, 3, 7, 7]}],  # 两类部分重叠
    "DET-05": [{"class": "bag", "box": [1, 1, 5, 6]}],  # 单背包场景
    "DET-06": [{"class": "person", "box": [2, 0, 7, 8]}],  # 单人场景
}  # 结束六幅图真值
predictions = [  # 定义含重复、误检和正确框的检测输出
    {"image": "DET-01", "class": "person", "score": 0.98, "box": [0, 0, 6, 8]},  # 正确人员高分框
    {"image": "DET-01", "class": "person", "score": 0.96, "box": [0, 0, 6, 7]},  # 人员高分重复框
    {"image": "DET-01", "class": "bag", "score": 0.95, "box": [1, 1, 6, 7]},  # 与人重叠的正确背包框
    {"image": "DET-02", "class": "person", "score": 0.93, "box": [1, 0, 5, 8]},  # 正确人员框
    {"image": "DET-02", "class": "person", "score": 0.91, "box": [1, 1, 5, 8]},  # 人员重复框
    {"image": "DET-03", "class": "bag", "score": 0.89, "box": [2, 2, 6, 6]},  # 正确背包框
    {"image": "DET-04", "class": "person", "score": 0.86, "box": [0, 1, 4, 8]},  # 正确人员框
    {"image": "DET-04", "class": "bag", "score": 0.84, "box": [3, 3, 7, 7]},  # 正确背包框
    {"image": "DET-05", "class": "bag", "score": 0.80, "box": [1, 1, 5, 6]},  # 正确背包框
    {"image": "DET-06", "class": "person", "score": 0.77, "box": [2, 0, 7, 8]},  # 正确人员框
    {"image": "DET-06", "class": "bag", "score": 0.60, "box": [0, 0, 2, 2]},  # 背包误检
]  # 结束十一条检测预测
print("输入预览：image | ground_truth")  # 输出六幅图真值表头
for image_id, boxes in ground_truth.items():  # 遍历六个检测场景
    print(image_id, boxes)  # 展示类别与框坐标
print("预测：image | class | score | box")  # 输出模型检测表头
for prediction in predictions:  # 遍历十一条原始预测
    print(prediction)  # 展示重复框和误检

输入预览：image | ground_truth
DET-01 [{'class': 'person', 'box': [0, 0, 6, 8]}, {'class': 'bag', 'box': [1, 1, 6, 7]}]
DET-02 [{'class': 'person', 'box': [1, 0, 5, 8]}]
DET-03 [{'class': 'bag', 'box': [2, 2, 6, 6]}]
DET-04 [{'class': 'person', 'box': [0, 1, 4, 8]}, {'class': 'bag', 'box': [3, 3, 7, 7]}]
DET-05 [{'class': 'bag', 'box': [1, 1, 5, 6]}]
DET-06 [{'class': 'person', 'box': [2, 0, 7, 8]}]
预测：image | class | score | box
{'image': 'DET-01', 'class': 'person', 'score': 0.98, 'box': [0, 0, 6, 8]}
{'image': 'DET-01', 'class': 'person', 'score': 0.96, 'box': [0, 0, 6, 7]}
{'image': 'DET-01', 'class': 'bag', 'score': 0.95, 'box': [1, 1, 6, 7]}
{'image': 'DET-02', 'class': 'person', 'score': 0.93, 'box': [1, 0, 5, 8]}
{'image': 'DET-02', 'class': 'person', 'score': 0.91, 'box': [1, 1, 5, 8]}
{'image': 'DET-03', 'class': 'bag', 'score': 0.89, 'box': [2, 2, 6, 6]}
{'image': 'DET-04', 'class': 'person', 'score': 0.86, 'box': [0, 1, 4, 8]}
{'image': 'DET-04', 'class': 'bag', 'score': 0.84, '

## Baseline / 基线：不做 NMS 直接评估所有预测

同一 GT 只能匹配一次，因此第二个高 IoU 重复框会变成 FP。基线保留十一条预测并计算逐类 AP。

In [2]:
def box_iou(left, right):  # 手写两个半开坐标检测框 IoU
    intersection_left = max(left[0], right[0])  # 计算交集左边界
    intersection_top = max(left[1], right[1])  # 计算交集上边界
    intersection_right = min(left[2], right[2])  # 计算交集右边界
    intersection_bottom = min(left[3], right[3])  # 计算交集下边界
    intersection_width = max(0, intersection_right - intersection_left)  # 计算非负交集宽度
    intersection_height = max(0, intersection_bottom - intersection_top)  # 计算非负交集高度
    intersection = intersection_width * intersection_height  # 计算交集面积
    left_area = max(0, left[2] - left[0]) * max(0, left[3] - left[1])  # 计算左框面积
    right_area = max(0, right[2] - right[0]) * max(0, right[3] - right[1])  # 计算右框面积
    union = left_area + right_area - intersection  # 计算并集面积
    return intersection / union if union > 0 else 0.0  # 返回 IoU 并处理退化框
def evaluate_class(detections, class_name, iou_threshold=0.5):  # 对一个类别计算匹配序列与 AP
    class_detections = sorted([item for item in detections if item["class"] == class_name], key=lambda item: -item["score"])  # 按 score 降序过滤类别
    matched = {image_id: set() for image_id in ground_truth}  # 记录每幅图已匹配 GT 索引
    total_gt = sum(sum(box["class"] == class_name for box in boxes) for boxes in ground_truth.values())  # 统计当前类别真值数
    true_positive = []  # 保存排序预测 TP 标记
    false_positive = []  # 保存排序预测 FP 标记
    ledger = []  # 保存每条预测最佳 IoU 和匹配原因
    for detection in class_detections:  # 按分数顺序逐条匹配
        candidates = [(index, box_iou(detection["box"], target["box"])) for index, target in enumerate(ground_truth[detection["image"]]) if target["class"] == class_name and index not in matched[detection["image"]]]  # 计算同图同类未匹配 GT IoU
        best_index, best_iou = max(candidates, key=lambda item: item[1]) if candidates else (-1, 0.0)  # 选择最大 IoU 候选
        is_true = best_iou >= iou_threshold  # 判断是否达到匹配阈值
        if is_true:  # 处理成功匹配预测
            matched[detection["image"]].add(best_index)  # 标记该 GT 已被占用
        true_positive.append(1 if is_true else 0)  # 记录 TP 序列
        false_positive.append(0 if is_true else 1)  # 记录 FP 序列
        ledger.append((detection["image"], detection["score"], best_iou, "TP" if is_true else "FP"))  # 写入匹配账本
    cumulative_tp = []  # 计算累计 TP
    cumulative_fp = []  # 计算累计 FP
    for index in range(len(true_positive)):  # 遍历排序检测位置
        cumulative_tp.append(sum(true_positive[:index + 1]))  # 累加当前位置前 TP
        cumulative_fp.append(sum(false_positive[:index + 1]))  # 累加当前位置前 FP
    recall = [value / total_gt for value in cumulative_tp]  # 计算每个阈值召回率
    precision = [tp / (tp + fp) for tp, fp in zip(cumulative_tp, cumulative_fp)]  # 计算每个阈值精确率
    average_precision = 0.0  # 初始化插值 AP 面积
    previous_recall = 0.0  # 初始化上一个召回断点
    for recall_level in sorted(set(recall)):  # 遍历发生变化的召回率
        interpolated_precision = max(value for value, current_recall in zip(precision, recall) if current_recall >= recall_level)  # 计算右侧最大精确率包络
        average_precision += (recall_level - previous_recall) * interpolated_precision  # 累积召回增量对应面积
        previous_recall = recall_level  # 更新召回断点
    return average_precision, ledger, precision, recall  # 返回 AP、匹配账本和 PR 序列
baseline_ap = {}  # 保存不做 NMS 的两类 AP
for class_name in ["person", "bag"]:  # 分别评估人员和背包
    ap, ledger, precision, recall = evaluate_class(predictions, class_name)  # 计算原始预测 AP
    baseline_ap[class_name] = ap  # 保存当前类别基线指标
    print(class_name, "baseline ledger=", ledger, "AP=", round(ap, 4))  # 展示重复框如何成为 FP
baseline_map = sum(baseline_ap.values()) / len(baseline_ap)  # 计算两类平均 AP
print(f"no-NMS mAP@0.5={baseline_map:.4f}")  # 汇总不去重基线

person baseline ledger= [('DET-01', 0.98, 1.0, 'TP'), ('DET-01', 0.96, 0.0, 'FP'), ('DET-02', 0.93, 1.0, 'TP'), ('DET-02', 0.91, 0.0, 'FP'), ('DET-04', 0.86, 1.0, 'TP'), ('DET-06', 0.77, 1.0, 'TP')] AP= 0.75
bag baseline ledger= [('DET-01', 0.95, 1.0, 'TP'), ('DET-03', 0.89, 1.0, 'TP'), ('DET-04', 0.84, 1.0, 'TP'), ('DET-05', 0.8, 1.0, 'TP'), ('DET-06', 0.6, 0.0, 'FP')] AP= 1.0
no-NMS mAP@0.5=0.8750


## 核心实现：按图按类执行 NMS，并输出抑制轨迹

只有同图同类框互相抑制。保留框按 score 降序，IoU 超过 0.5 的后续框写入 suppression ledger。

In [3]:
def non_maximum_suppression(detections, iou_threshold=0.5, class_aware=True):  # 手写检测框 NMS
    kept = []  # 收集所有图最终保留预测
    suppression = []  # 收集被抑制框和原因
    for image_id in ground_truth:  # 逐幅图独立执行 NMS
        image_detections = [item for item in detections if item["image"] == image_id]  # 读取当前图预测
        remaining = sorted(image_detections, key=lambda item: -item["score"])  # 按置信度降序初始化队列
        while remaining:  # 直到当前图所有框处理完成
            selected = remaining.pop(0)  # 取最高分框进入保留集
            kept.append(selected)  # 保存当前选择框
            survivors = []  # 收集不应被当前框抑制的预测
            for candidate in remaining:  # 遍历剩余低分框
                same_group = candidate["class"] == selected["class"] if class_aware else True  # 决定是否按类别隔离抑制
                overlap = box_iou(selected["box"], candidate["box"])  # 计算候选与选中框 IoU
                if same_group and overlap > iou_threshold:  # 检查是否满足抑制条件
                    suppression.append((image_id, selected["class"], selected["score"], candidate["class"], candidate["score"], overlap))  # 写入抑制账本
                else:  # 处理不同类或低重叠候选
                    survivors.append(candidate)  # 保留候选进入后续 NMS
            remaining = survivors  # 更新当前图待处理队列
    return kept, suppression  # 返回保留预测与抑制轨迹
class_aware_detections, class_aware_suppression = non_maximum_suppression(predictions, class_aware=True)  # 执行正确按类 NMS
print("image | kept_class/score | suppressed_class/score | IoU")  # 输出抑制账本表头
for item in class_aware_suppression:  # 遍历真实重复框抑制事件
    print(f"{item[0]} | {item[1]}/{item[2]:.2f} | {item[3]}/{item[4]:.2f} | {item[5]:.3f}")  # 展示同类高重叠去重
print("NMS 前后预测数：", len(predictions), "->", len(class_aware_detections))  # 展示候选压缩效果

image | kept_class/score | suppressed_class/score | IoU
DET-01 | person/0.98 | person/0.96 | 0.875
DET-02 | person/0.93 | person/0.91 | 0.875
NMS 前后预测数： 11 -> 9


## NMS 后逐类 AP、PR 与 mAP

In [4]:
nms_ap = {}  # 保存按类 NMS 后 AP
print("class | AP_before | AP_after | precision | recall")  # 输出检测评估结果表头
for class_name in ["person", "bag"]:  # 分别评估两类目标
    ap, ledger, precision, recall = evaluate_class(class_aware_detections, class_name)  # 对去重预测执行一对一匹配
    nms_ap[class_name] = ap  # 保存当前类别 AP
    print(f"{class_name:6} | {baseline_ap[class_name]:.4f} | {ap:.4f} | {[round(value, 3) for value in precision]} | {[round(value, 3) for value in recall]}")  # 展示完整 PR 轨迹
    print("  match ledger：", ledger)  # 展示逐 score TP/FP 判断
nms_map = sum(nms_ap.values()) / len(nms_ap)  # 计算两类 NMS 后 mAP
print(f"mAP@0.5：no-NMS={baseline_map:.4f}，class-aware NMS={nms_map:.4f}")  # 对比同一预测集指标

class | AP_before | AP_after | precision | recall
person | 0.7500 | 1.0000 | [1.0, 1.0, 1.0, 1.0] | [0.25, 0.5, 0.75, 1.0]
  match ledger： [('DET-01', 0.98, 1.0, 'TP'), ('DET-02', 0.93, 1.0, 'TP'), ('DET-04', 0.86, 1.0, 'TP'), ('DET-06', 0.77, 1.0, 'TP')]
bag    | 1.0000 | 1.0000 | [1.0, 1.0, 1.0, 1.0, 0.8] | [0.25, 0.5, 0.75, 1.0, 1.0]
  match ledger： [('DET-01', 0.95, 1.0, 'TP'), ('DET-03', 0.89, 1.0, 'TP'), ('DET-04', 0.84, 1.0, 'TP'), ('DET-05', 0.8, 1.0, 'TP'), ('DET-06', 0.6, 0.0, 'FP')]
mAP@0.5：no-NMS=0.8750，class-aware NMS=1.0000


## 失败案例与修正：类别无关 NMS 误删重叠背包

DET-01 的 person 与 bag IoU 超过 0.5，但它们是两个合法对象。类别无关 NMS 会因 person 分数更高而删除 bag；按类 NMS 两者都保留。

In [5]:
class_agnostic_detections, class_agnostic_suppression = non_maximum_suppression(predictions, class_aware=False)  # 复现跨类别错误抑制
agnostic_det01 = [(item["class"], item["score"]) for item in class_agnostic_detections if item["image"] == "DET-01"]  # 提取错误策略首图结果
aware_det01 = [(item["class"], item["score"]) for item in class_aware_detections if item["image"] == "DET-01"]  # 提取正确按类首图结果
cross_class_overlap = box_iou(predictions[0]["box"], predictions[2]["box"])  # 计算首图 person 与 bag 框 IoU
agnostic_bag_ap, agnostic_bag_ledger, agnostic_precision, agnostic_recall = evaluate_class(class_agnostic_detections, "bag")  # 评估错误 NMS 对背包召回影响
print(f"DET-01 person/bag IoU={cross_class_overlap:.3f}")  # 展示跨类别高重叠条件
print("class-agnostic kept：", agnostic_det01)  # 展示背包被错误删除
print("class-aware kept：", aware_det01)  # 展示两类合法对象同时保留
print(f"bag AP：agnostic={agnostic_bag_ap:.4f}，aware={nms_ap['bag']:.4f}")  # 量化跨类抑制影响
print("agnostic suppression：", [item for item in class_agnostic_suppression if item[0] == "DET-01"])  # 展示错误抑制证据

DET-01 person/bag IoU=0.625
class-agnostic kept： [('person', 0.98)]
class-aware kept： [('person', 0.98), ('bag', 0.95)]
bag AP：agnostic=0.7500，aware=1.0000
agnostic suppression： [('DET-01', 'person', 0.98, 'person', 0.96, 0.875), ('DET-01', 'person', 0.98, 'bag', 0.95, 0.625)]


## 结果解读

重复框在一对一匹配中会成为 FP，按类 NMS 删除它们后 PR 曲线和 mAP 改善。DET-01 说明 NMS 分组键至少应包含 image 和 class；若模型采用 class-agnostic box regression，也不能无条件跨类抑制。

## 生产边界

教学评估只有两类六图和 IoU 0.5。生产 COCO mAP 需平均多个 IoU 阈值并处理 crowd、area range 和 max detections；框坐标还要明确 resize/letterbox 的逆变换。NMS 可替换为 soft-NMS 或 learned NMS，但必须按部署实现复算指标并保存置信度阈值。

## 最小回归测试

In [6]:
assert len(ground_truth) >= 6 and len(predictions) >= 6  # 保证检测案例覆盖多幅图和预测
assert 0.0 <= cross_class_overlap <= 1.0 and cross_class_overlap > 0.5  # 保证跨类别高重叠反例成立
assert len(class_aware_detections) < len(predictions)  # 保证 NMS 真实抑制重复框
assert nms_map > baseline_map  # 保证去重后同预测集 mAP 提升
assert "bag" not in [class_name for class_name, score in agnostic_det01]  # 保证类别无关 NMS 误删背包
assert {class_name for class_name, score in aware_det01} == {"person", "bag"}  # 保证按类 NMS 保留两类对象
assert nms_ap["bag"] > agnostic_bag_ap  # 保证修正跨类抑制改善背包 AP